# Classical Model Evaluation

This notebook displays the evaluation results generated by `classical/training/train_models.py`. It creates model comparison figures and diagnostic plots.

**SVR setting:** `linear_high_c` with `kernel="linear"`, `C=500.0`, and `epsilon=1.0`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = os.getcwd()
if os.path.basename(current_dir).lower() == 'analysis':
    project_dir = os.path.dirname(os.path.dirname(current_dir))
else:
    project_dir = current_dir

result_dir = os.path.join(project_dir, 'artifacts', 'classical', 'training')
figure_dir = os.path.join(result_dir, 'figures')
os.makedirs(figure_dir, exist_ok=True)

metrics = pd.read_csv(os.path.join(result_dir, 'metrics_v3.csv'))
predictions = pd.read_csv(os.path.join(result_dir, 'predictions_v3.csv'))

model_columns = {
    'Linear Regression': 'linear_regression_prediction',
    'Ridge Regression': 'ridge_regression_prediction',
    'Support Vector Regression': 'support_vector_regression_prediction',
}

colors = {
    'Linear Regression': '#1f77b4',
    'Ridge Regression': '#2ca02c',
    'Support Vector Regression': '#d62728',
}

metrics

## Model Performance Table

In [ ]:
display_metrics = metrics.copy()
display_metrics['R2'] = display_metrics['R2'].map(lambda x: f'{x:.4f}')
display_metrics['MAE'] = display_metrics['MAE'].map(lambda x: f'{x:.2f}')
display_metrics['RMSE'] = display_metrics['RMSE'].map(lambda x: f'{x:.2f}')
display_metrics['MAPE'] = display_metrics['MAPE'].map(lambda x: f'{x * 100:.2f}%')
display_metrics

## Combined Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(predictions['Year'], predictions['actual_yield_kg_ha'], marker='o', linewidth=2.6, color='black', label='Actual Yield')

for model_name, pred_col in model_columns.items():
    row = metrics.loc[metrics['model'] == model_name].iloc[0]
    label = f"{model_name} (RMSE={row['RMSE']:.2f}, MAPE={row['MAPE'] * 100:.2f}%)"
    ax.plot(predictions['Year'], predictions[pred_col], marker='s', linestyle='--', linewidth=1.9, color=colors[model_name], label=label)

ax.set_title('Actual and Predicted Barley Yield on Test Years')
ax.set_xlabel('Year')
ax.set_ylabel('Yield (kg/ha)')
ax.legend()
plt.tight_layout()
path = os.path.join(figure_dir, 'model_prediction_comparison_v3.png')
plt.savefig(path, dpi=300, bbox_inches='tight')
plt.show()
path

## Three-Panel Prediction Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, (model_name, pred_col) in zip(axes, model_columns.items()):
    row = metrics.loc[metrics['model'] == model_name].iloc[0]
    ax.plot(predictions['Year'], predictions['actual_yield_kg_ha'], marker='o', linewidth=2.3, color='black', label='Actual')
    ax.plot(predictions['Year'], predictions[pred_col], marker='s', linestyle='--', linewidth=2.0, color=colors[model_name], label='Predicted')
    ax.set_title(f"{model_name}\nRMSE={row['RMSE']:.2f}, MAPE={row['MAPE'] * 100:.2f}%")
    ax.set_xlabel('Year')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='best')

axes[0].set_ylabel('Yield (kg/ha)')
fig.suptitle('Model-wise Actual vs Predicted Yield Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
path = os.path.join(figure_dir, 'model_prediction_three_panel_v3.png')
plt.savefig(path, dpi=300, bbox_inches='tight')
plt.show()
path

## Scatter Fit Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
actual = predictions['actual_yield_kg_ha']
min_value = actual.min()
max_value = actual.max()

for model_name, pred_col in model_columns.items():
    pred = predictions[pred_col]
    ax.scatter(actual, pred, s=75, alpha=0.82, color=colors[model_name], edgecolor='white', linewidth=0.8, label=model_name)
    slope, intercept = np.polyfit(actual, pred, 1)
    x_line = np.linspace(min_value, max_value, 100)
    ax.plot(x_line, slope * x_line + intercept, color=colors[model_name], linestyle=':', linewidth=1.8)

all_pred_values = np.concatenate([predictions[col].to_numpy() for col in model_columns.values()])
plot_min = min(min_value, all_pred_values.min())
plot_max = max(max_value, all_pred_values.max())
ax.plot([plot_min, plot_max], [plot_min, plot_max], linestyle='--', color='gray', linewidth=1.6, label='Ideal fit')
ax.set_xlim(plot_min - 40, plot_max + 40)
ax.set_ylim(plot_min - 40, plot_max + 40)
ax.set_title('Predicted vs Actual Yield Scatter Fit')
ax.set_xlabel('Actual Yield (kg/ha)')
ax.set_ylabel('Predicted Yield (kg/ha)')
ax.legend()
plt.tight_layout()
path = os.path.join(figure_dir, 'prediction_scatter_fit_v3.png')
plt.savefig(path, dpi=300, bbox_inches='tight')
plt.show()
path

## Annual Error Percentage

In [ ]:
error_rows = []
for model_name, pred_col in model_columns.items():
    error_pct = (predictions[pred_col] - predictions['actual_yield_kg_ha']).abs() / predictions['actual_yield_kg_ha'] * 100
    for year, value in zip(predictions['Year'], error_pct):
        error_rows.append({'Year': year, 'Model': model_name, 'Absolute Percentage Error': value})

error_df = pd.DataFrame(error_rows)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=error_df, x='Year', y='Absolute Percentage Error', hue='Model', palette=colors, ax=ax)
ax.set_title('Annual Absolute Percentage Error by Model')
ax.set_xlabel('Year')
ax.set_ylabel('Absolute Percentage Error (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Model')
plt.tight_layout()
path = os.path.join(figure_dir, 'annual_error_percentage_v3.png')
plt.savefig(path, dpi=300, bbox_inches='tight')
plt.show()
path

## Diagnostic Panel Function

In [ ]:
def plot_diagnostic_panel(model_name, y_true, y_pred, years, save_dir):
    residuals = y_true - y_pred
    abs_pct_error = residuals.abs() / y_true * 100
    metric_row = metrics.loc[metrics['model'] == model_name].iloc[0]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{model_name} Diagnostic Panel', fontsize=16, fontweight='bold')

    axes[0, 0].plot(years, y_true, marker='o', linewidth=2.3, label='Actual', color='black')
    axes[0, 0].plot(years, y_pred, marker='s', linestyle='--', linewidth=2.0, label='Predicted', color=colors[model_name])
    axes[0, 0].set_title('Actual vs Predicted Over Time')
    axes[0, 0].set_xlabel('Year')
    axes[0, 0].set_ylabel('Yield (kg/ha)')
    axes[0, 0].legend()

    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())
    axes[0, 1].scatter(y_true, y_pred, s=75, color=colors[model_name], edgecolor='white', linewidth=0.8)
    axes[0, 1].plot([min_value, max_value], [min_value, max_value], linestyle='--', color='gray', linewidth=1.5)
    axes[0, 1].set_title('Predicted vs Actual')
    axes[0, 1].set_xlabel('Actual Yield (kg/ha)')
    axes[0, 1].set_ylabel('Predicted Yield (kg/ha)')

    axes[1, 0].axhline(0, color='gray', linestyle='--', linewidth=1.4)
    axes[1, 0].bar(years, residuals, color=colors[model_name], alpha=0.78)
    axes[1, 0].set_title('Residuals by Year')
    axes[1, 0].set_xlabel('Year')
    axes[1, 0].set_ylabel('Actual - Predicted')

    axes[1, 1].bar(years, abs_pct_error, color=colors[model_name], alpha=0.78)
    axes[1, 1].set_title('Annual Absolute Percentage Error')
    axes[1, 1].set_xlabel('Year')
    axes[1, 1].set_ylabel('Error (%)')

    metric_text = (
        f"R2: {metric_row['R2']:.4f}\n"
        f"MAE: {metric_row['MAE']:.2f}\n"
        f"RMSE: {metric_row['RMSE']:.2f}\n"
        f"MAPE: {metric_row['MAPE'] * 100:.2f}%"
    )
    fig.text(0.78, 0.92, metric_text, fontsize=11, bbox=dict(boxstyle='round', facecolor='white', alpha=0.88))

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    file_name = model_name.lower().replace(' ', '_') + '_diagnostic_panel_v3.png'
    output_path = os.path.join(save_dir, file_name)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    return output_path

## Individual Diagnostic Panels

In [ ]:
saved_panel_paths = []
y_true = predictions['actual_yield_kg_ha']
years = predictions['Year']

for model_name, pred_col in model_columns.items():
    saved_panel_paths.append(
        plot_diagnostic_panel(
            model_name=model_name,
            y_true=y_true,
            y_pred=predictions[pred_col],
            years=years,
            save_dir=figure_dir,
        )
    )

saved_panel_paths